# ER1: No-Comm Control Baseline

**Objective:** Establish the performance floor by training agents with **zero communication** on the Discovery K-N convergence task. Any communication method (ER2-E1) must beat this to justify its complexity.

**What this isolates:** Pure task difficulty under partial observability — agents can only sense via LiDAR, with no message passing between them.

**How to use this notebook:**
1. Set `PROFILE = "fast"` for a quick validation run (~5 min, 1 config, 1 seed) or `"complete"` for the full sweep
2. Run all cells top to bottom
3. Results are saved to `results/er1/<run_id>/` with structured subfolders (input, logs, output, report)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment
from src.display import display_config, display_metrics, display_sweep_summary, display_run_status
from src.storage import ExperimentStorage
from src.runner import run_sweep, evaluate_with_vmas, make_heuristic_policy_fn
from src.plotting import plot_sweep_heatmap, plot_seed_variance, save_figure

print(f"Torch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

## 2. Configuration

Set `PROFILE` below to control experiment scope:
- **`"fast"`**: 1 configuration (4 agents, MAPPO, seed 0), 120K training frames. Use this for quick validation.
- **`"complete"`**: Full sweep — N ∈ {2,3,4,6} × lidar ∈ {0.25, 0.35, 0.45} × {MAPPO, IPPO} × 5 seeds = **120 runs**.

All parameters are loaded from `configs/er1_no_comm.yaml` and the profile applies overrides on top.

In [ ]:
PROFILE = "fast"  # "fast" or "complete"

spec = load_experiment(RENDEZVOUS_ROOT / "configs" / "er1_no_comm.yaml", profile=PROFILE)
spec.ensure_dirs()

display_config(spec, profile=PROFILE)

## 3. Sanity Check — Heuristic & Random Baselines

Before training, we verify the environment works and establish two reference points:
- **Heuristic**: The discovery scenario has a built-in `HeuristicPolicy` (circular patrol + greedy target-seeking). This is the non-learned upper reference.
- **Random**: Agents take random actions. This is the absolute floor.

We use `targets_respawn=False` so that episodes end when all targets are covered, making M1 (success rate) and M3 (steps to completion) meaningful.

### Metrics Reference
| Metric | ID | Meaning |
|--------|-----|---------|
| Success Rate | M1 | Fraction of episodes where **all** targets were covered |
| Avg Targets/Step | M1b | Mean targets covered per step (useful when `targets_respawn=True`) |
| Avg Return | M2 | Mean cumulative reward per episode |
| Avg Steps | M3 | Mean steps until done (`max_steps` if episode didn't complete) |
| Collisions/Episode | M4 | Mean agent-agent collisions per episode |
| Tokens/Episode | M5 | Communication tokens used (always 0 for ER1 — no comm) |

In [ ]:
SANITY_OVERRIDES = {
    "n_agents": 4, "n_targets": 7, "agents_per_target": 2,
    "targets_respawn": False,
}

heuristic_fn = make_heuristic_policy_fn()
heuristic_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=heuristic_fn, n_eval_episodes=200, n_envs=200,
)
display_metrics(heuristic_metrics, title="Heuristic Baseline (4 agents, 7 targets, K=2)")

random_metrics = evaluate_with_vmas(
    spec.task, task_overrides=SANITY_OVERRIDES,
    policy_fn=None, n_eval_episodes=200, n_envs=200,
)
display_metrics(random_metrics, title="Random Baseline (4 agents, 7 targets, K=2)")

## 4. Training

Runs the parameter sweep defined by the config + profile. Each run:
1. Saves a frozen config snapshot to `input/config.yaml`
2. Trains the policy using BenchMARL, logging to `logs/run.log`
3. Evaluates the trained policy (200 episodes, deterministic mode)
4. Saves metrics to `output/metrics.json`
5. Generates a human-readable `report.txt`

BenchMARL raw outputs (CSV training curves, checkpoints) go to `output/benchmarl/`.

Use `skip_complete=True` (default) to safely resume interrupted sweeps — completed runs are detected and skipped.

In [ ]:
display_run_status(spec)

results = run_sweep(spec, skip_complete=True)

## 5. Results

Each completed run produces the following output structure:
```
results/er1/<run_id>/
  input/
    config.yaml           # Frozen config: all task, training, and sweep params
  logs/
    run.log               # Full training log (timestamps, metrics, errors)
    training_log.csv      # Per-iteration training metrics
  output/
    benchmarl/            # Raw BenchMARL artifacts (CSV curves, checkpoints, JSON log)
    metrics.json          # Final evaluation metrics (M1-M5)
  report.txt              # Human-readable summary of config + results
```

The key metrics in `metrics.json`:
- **M1 (Success Rate):** Did agents manage to cover all targets? Higher = better coordination.
- **M2 (Avg Return):** Cumulative reward — balances covering rewards, collision penalties, and time penalties.
- **M3 (Avg Steps):** How quickly agents complete the task. Lower = more efficient (max_steps if never completed).
- **M4 (Collisions):** Agent-agent collisions. Lower = better spatial awareness.
- **M5 (Tokens):** Always 0 for ER1 (no communication).

In [ ]:
storage = ExperimentStorage("er1")
all_metrics = storage.load_all_metrics()

if not all_metrics:
    print("No completed runs yet. Run section 4 first.")
else:
    print(f"Loaded {len(all_metrics)} completed runs\n")
    display_sweep_summary(all_metrics)

    # Show the report from the latest run
    latest_run_id = sorted(all_metrics.keys())[-1]
    report_path = storage.results_dir / latest_run_id / "report.txt"
    if report_path.exists():
        print(f"\n{'=' * 70}")
        print(f"  Latest run report: {latest_run_id}")
        print(f"{'=' * 70}")
        print(report_path.read_text())

## 6. Analysis

Visualizations to understand the results:
- **6a. Heatmap:** Success rate across agent count and LiDAR range — shows which configurations are easiest/hardest.
- **6b. Algorithm comparison:** MAPPO vs IPPO with seed variance — checks if one algorithm consistently outperforms.
- **6c. Summary table:** All core metrics grouped by algorithm with mean and standard deviation.

These plots are saved to the experiment results directory for use in papers/reports.

In [ ]:
df = storage.to_dataframe()

if df.empty:
    print("No data for plots. Complete section 4 first.")
else:
    # 6a. Success rate heatmap
    if len(df["n_agents"].unique()) > 1 if "n_agents" in df.columns else False:
        fig = plot_sweep_heatmap(
            df, metric="M1_success_rate",
            row_param="n_agents", col_param="lidar_range",
            title="ER1 No-Comm: Success Rate (N agents x LiDAR range)",
        )
        save_figure(fig, str(spec.results_dir / "er1_success_heatmap.png"))
        plt.show()
    else:
        print("Heatmap requires multiple agent counts (use 'complete' profile).")

In [ ]:
if not df.empty:
    # 6b. Algorithm comparison
    if "algorithm" in df.columns and len(df["algorithm"].unique()) > 1:
        fig = plot_seed_variance(
            df, metric="M1_success_rate", group_by="algorithm",
            title="ER1: MAPPO vs IPPO Success Rate (across seeds)",
        )
        save_figure(fig, str(spec.results_dir / "er1_algo_comparison.png"))
        plt.show()
    else:
        print("Algorithm comparison requires multiple algorithms (use 'complete' profile).")

    # 6c. Summary table
    print("\n" + "=" * 70)
    print("  SUMMARY TABLE: Core Metrics by Algorithm")
    print("=" * 70)
    group_cols = [c for c in ["algorithm"] if c in df.columns]
    if group_cols:
        summary = df.groupby(group_cols).agg({
            "M1_success_rate": ["mean", "std"],
            "M2_avg_return": ["mean", "std"],
            "M3_avg_steps": ["mean", "std"],
            "M4_avg_collisions": ["mean", "std"],
        }).round(4)
        from IPython.display import display
        display(summary)
    else:
        for key in ["M1_success_rate", "M2_avg_return", "M3_avg_steps", "M4_avg_collisions"]:
            vals = df[key].values
            print(f"  {key:<35} {vals.mean():.4f} +/- {vals.std():.4f}")

## 7. Pass/Fail Assessment

**Expected outcome:** The no-comm floor should be in the **20-50% success range** for N=4, K=2. This means:
- If success is **below 20%**: The task may be too hard for any method — consider easier configs.
- If success is **20-50%**: Floor established. Communication methods (ER2-E1) must beat this.
- If success is **above 50%**: Task may be too easy without comm — consider harder configs (more targets, smaller LiDAR).

This assessment determines whether the experimental design is sound before investing compute in communication experiments.

In [ ]:
if not df.empty:
    # Average success across all runs (or filter to N=4 if available)
    if "n_agents" in df.columns:
        n4_df = df[df["n_agents"] == 4]
    else:
        n4_df = df

    if not n4_df.empty:
        avg_success = n4_df["M1_success_rate"].mean()
        avg_return = n4_df["M2_avg_return"].mean()
        avg_steps = n4_df["M3_avg_steps"].mean()

        print("=" * 70)
        print(f"  N=4, K=2 Average Success Rate:  {avg_success:.1%}")
        print(f"  N=4, K=2 Average Return:        {avg_return:.2f}")
        print(f"  N=4, K=2 Average Steps:         {avg_steps:.0f}")
        print("=" * 70)

        if 0.20 <= avg_success <= 0.50:
            print("\n  PASS: Floor established in expected range (20-50%).")
            print("  Proceed to ER2+ experiments. Comm methods must beat this floor.")
        elif avg_success > 0.50:
            print("\n  NOTE: Floor higher than expected (>50%). Consider:")
            print("    - Increasing n_targets or agents_per_target")
            print("    - Reducing lidar_range")
        else:
            print(f"\n  NOTE: Floor lower than expected (<20%). Consider:")
            print("    - Training for more frames (current: {:,})".format(spec.train.max_n_frames))
            print("    - Checking if the task is too hard for this config")
            if PROFILE == "fast":
                print("    - Running with PROFILE='complete' for full training")
    else:
        print("No N=4 runs found.")
else:
    print("No data available. Run training first (section 4).")